<a href="https://colab.research.google.com/github/bah862696-coder/DI-Bootcamp/blob/master/Exercises_XP_MCP_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercises XP: Minimal MCP over STDIO (Student)

Build a tiny MCP server and client that talk over STDIO. This code is supposed to be executed in a local jupyter notebook not Colab's notebook.

## What you'll learn
- How MCP structures hosts/clients/servers and why STDIO is great locally.
- How to register a tool (action) and a resource (read-only context) on a server.
- How to write a client that initializes, lists, and invokes those features.

## Setup
Run the install cell, then restart the runtime if Colab asks. Python 3.10+ required.

In [ ]:
# Install MCP CLI + SDK
%pip install -qU "mcp[cli]"

In [ ]:
# Quick verify
!python --version
!mcp --help | head -n 5

## A. Server (server.py)
Create a small MCP server named "Demo" with:
- Tool `add(a: int, b: int) -> int` returning the sum.
- Resource template `greeting://{name}` returning "Hello, {name}!".
- Start the STDIO loop in `__main__`.

In [1]:
%%writefile server.py
from mcp.server.fastmcp import FastMCP

# Création du serveur nommé "Demo"
mcp = FastMCP("Demo")

@mcp.tool()
def add(a: int, b: int) -> int:
    """Return the sum of two integers."""
    # Retourne la somme de a et b
    return a + b

@mcp.resource("greeting://{name}")
def greet(name: str) -> str:
    """Return a greeting for the given name."""
    # Retourne la chaîne formatée
    return f"Hello, {name}!"

if __name__ == "__main__":
    # Démarrage explicite de la boucle STDIO
    mcp.run()

Writing server.py


## B. Client (client.py)
Write a client that:
1) Spawns the server via STDIO using the MCP CLI.
2) Initializes a session.
3) Lists resources and tools, printing their names.
4) Reads `greeting://hello` and prints it.
5) Calls tool `add` with a=1, b=7 and prints the result.

In [2]:
%%writefile client.py
import asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

# Configuration pour lancer le serveur via l'interface CLI mcp
server_params = StdioServerParameters(command="mcp", args=["run", "server.py"], env=None)

def extract_content(payload):
    """Meilleur effort pour extraire le texte des réponses MCP."""
    if hasattr(payload, "contents"):
        contents = payload.contents
        if contents:
            first = contents[0]
            if hasattr(first, "text"):
                return first.text
            if isinstance(first, dict) and "text" in first:
                return first["text"]
            return str(first)
    if hasattr(payload, "content"):
        return payload.content
    return str(payload)

async def run():
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            # 2) Initialisation de la session
            await session.initialize()

            # 3) Lister les ressources et imprimer leurs noms
            resources = await session.list_resources()
            print("Resources:")
            for resource in resources.resources:
                print(f"- {resource.name}")

            # 3) Lister les outils et imprimer leurs noms
            tools = await session.list_tools()
            print("\nTools:")
            for tool in tools.tools:
                print(f"- {tool.name}")

            # 4) Lire greeting://hello et imprimer le contenu
            print("\nReading greeting://hello")
            greeting = await session.read_resource("greeting://hello")
            print(extract_content(greeting))

            # 5) Appeler l'outil add avec a=1, b=7 et imprimer le résultat
            print("\nCalling add(1,7)")
            result = await session.call_tool("add", arguments={"a": 1, "b": 7})
            print(extract_content(result))

if __name__ == "__main__":
    asyncio.run(run())

Writing client.py


## C. Run
One terminal (client spawns server):
```
python client.py
```

Or two terminals:
```
mcp run server.py
python client.py
```

In Colab, run the next cell (client will spawn the server automatically).

In [ ]:
# Run the client (spawns the server over STDIO)
!python client.py

## Troubleshooting
- `mcp: command not found` ? rerun the install cell or restart runtime.
- Connection closed ? open a second terminal and run `mcp run server.py` to check server errors.
- Type errors ? ensure JSON args are ints for `add`.